# Worst-case capacity: climate chamber and module room

Geometry and envelope are fixed for both rooms now, with one exception: the
module room's facade is still a scenario building element, not a settled
construction, so it stays a variable here rather than a fixed input.

Two separate questions, because the two rooms are settled to different
degrees:

| room | what's fixed | what this notebook resolves |
| --- | --- | --- |
| Climate chamber | envelope, setpoints (0-45 °C), ramp (8 h) | the worst-case steady load: ACH, ventilation air source, and an exposed concrete pad |
| Module room | envelope *except the facade*, setpoints (18-35 °C) | capacity as a function of ramp time, worst case over the facade |

Both machines are still sized against an unlimited tank (no buffering of the
ramp surge), so the numbers below are the room's design capacity, same as the
sensitivity notebook.

**Since 2026-09-23 the design capacity is the larger of two modes, never their
sum.** *Operating* is the steady hold during an experiment. *Ramp* is charging
the mass plus the hold at the far setpoint, at the same extreme boundary but
with nobody inside and the Artificial Sun off. Every figure below shows both.

## Decisions carried into this run

Settled this session, not swept:

- **Climate chamber ACH: 1.0**, not because it is occupied — it never is — but as a standing refresh rate rather than sealing the room entirely.
- **Ventilation air is pre-conditioned to 18 °C**, not drawn raw from outdoors and not at the previous 21 °C assumption. Applied to both rooms, since the 21 °C figure was a shared placeholder in `study/rooms.yaml`, not something specific to one room.
- **Climate chamber occupants: 0.** `rooms.py`'s generic default carries 2 occupants; the chamber is never occupied, so their sensible and latent gain is removed rather than inherited by accident.
- **Exposed thermal mass in the chamber: a 100 mm dense-concrete pad, 12 m².** Using the `dense_concrete` preset already in `params.py` (ρc = 2100 kJ/m³K, k = 1.80 W/mK) rather than inventing new material properties.
- **Climate chamber ramp: 480 min (8 h), fixed.** Confirmed separately as the fastest ramp the room commits to.
- **Climate chamber heating uses the Sun-off scenario, cooling uses Sun-on** — for the *operating* mode, each duty keeps its own worst case. The *ramp* mode is Sun-off by definition (decided 2026-09-23), so it is the same in both.
- **Radiant: hung ceiling panels, 20 % of floor + ceiling area** (decided 2026-09-23). They carry load up to their limit in both modes and do not change any room total here; they only move load from the air coil to the panels.
- **Module room: no added lining mass in this pass** (kept at zero). You mentioned added mass is moving from an absolute m² figure to a percentage of interior area — that's a `params.py`/`rooms.yaml` change of its own and isn't made here; see Open items.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from zcbsl_resize import physics, rooms, study  # noqa: E402
from zcbsl_resize.params import MASS_PRESETS  # noqa: E402

FIGURES = REPO / "notebooks" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# Same house style as 01_sensitivity.ipynb, so the two notebooks read as one set.
EARTH = ["#8a6f4e", "#6f7f5c", "#a3705c", "#5f7480", "#9c8a5a", "#7a6070"]
INK = "#2e2a26"
PAPER = "#f9f9f9"
MUTED = "#9a9188"

mpl.rcParams.update({
    "figure.facecolor": PAPER,
    "axes.facecolor": PAPER,
    "savefig.facecolor": PAPER,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "axes.titlesize": 10,
    "axes.titleweight": "semibold",
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "legend.frameon": False,
    "font.size": 9,
    "grid.color": "#e3ddd3",
    "grid.linewidth": 0.6,
    "figure.dpi": 110,
    "pdf.fonttype": 42,
})

def save(fig, index, slug):
    path = FIGURES / f"fig_{index}_{slug}.pdf"
    fig.savefig(path, bbox_inches="tight")
    print("wrote", path.relative_to(REPO))
    return path

def kw(x):
    return np.asarray(x, dtype=float) / 1000.0

## Climate chamber: worst-case steady load

The chamber's envelope, geometry and boundary conditions come straight from `rooms.climate_chamber()` -- nothing there is a free variable any more. What's added on top is this session's worst case: ACH, ventilation source temperature, zero occupants, and the concrete pad.

`boundary_temp_summer = 45°C` is now confirmed as the room's real design condition (climate-change future-proofing, not the historical 32°C Zurich figure) and `rooms.py` carries it directly. The chart below still shows 32°C alongside it, labelled as a superseded historical reference, purely so the size of the correction is visible -- it is no longer an open question.

In [2]:
concrete = MASS_PRESETS["dense_concrete"]

# 12 m2 of exposed concrete pad, expressed as this room's own
# added_mass_coverage percent -- added_mass_area no longer exists as a
# settable input, so this is how a real, physical pad size gets stated.
pad_area_m2 = 12.0
pad_coverage_pct = pad_area_m2 / rooms.climate_chamber().interior_area * 100.0

chamber_base = rooms.climate_chamber().replace(
    setpoint_min=0.0, setpoint_max=45.0,
    ramp_minutes=480.0,
    ach=1.0,
    vent_supply_temp=18.0,
    occupants=0.0,
    added_mass_coverage=pad_coverage_pct,
    added_mass_thickness=0.10,
    added_mass_rho_c=concrete["rho_c"],
    added_mass_k=concrete["k"],
)

def mode(r, duty):
    return "ramp" if bool(r[f"{duty}_set_by_ramp"]) else "operating"

rows = []
for summer in (32.0, 45.0):
    heating = physics.compute(chamber_base.replace(boundary_temp_summer=summer, equipment_w_per_m2=0.0))
    cooling = physics.compute(chamber_base.replace(boundary_temp_summer=summer, equipment_w_per_m2=chamber_base.equipment_w_per_m2))
    rows.append({
        "boundary_temp_summer": summer,
        "heating_operating_kw": float(kw(heating["heating_operating"])),
        "heating_ramp_kw": float(kw(heating["heating_ramp"])),
        "heating_design_kw": float(kw(heating["heating_design"])),
        "heating_set_by": mode(heating, "heating"),
        "electric_heating_kw": float(kw(heating["electric_heating"])),
        "cop_heating": float(heating["cop_heating"]),
        "cooling_operating_kw": float(kw(cooling["cooling_operating"])),
        "cooling_ramp_kw": float(kw(cooling["cooling_ramp"])),
        "cooling_design_kw": float(kw(cooling["cooling_design"])),
        "cooling_set_by": mode(cooling, "cooling"),
        "electric_cooling_kw": float(kw(cooling["electric_cooling"])),
        "cop_cooling": float(cooling["cop_cooling"]),
        "design_flow_ls_heat": float(heating["design_flow_ls"]),
        "design_flow_ls_cool": float(cooling["design_flow_ls"]),
        "latent_design_kw_cool": float(kw(cooling["latent_design"])),
    })

chamber_table = pd.DataFrame(rows).set_index("boundary_temp_summer")
chamber_table.round(2).T

boundary_temp_summer,32.0,45.0
heating_operating_kw,23.83,23.83
heating_ramp_kw,35.36,35.36
heating_design_kw,35.36,35.36
heating_set_by,ramp,ramp
electric_heating_kw,7.01,7.01
cop_heating,5.05,5.05
cooling_operating_kw,32.24,35.9
cooling_ramp_kw,31.69,35.35
cooling_design_kw,32.24,35.9
cooling_set_by,operating,operating


### Is the 8-hour ramp actually reachable with the pad added?

Worth checking explicitly: unlike the chamber's baseline sizing (zero added mass), this scenario now has 12 m² of concrete in the room, and added mass is what makes `fastest_ramp_minutes` diverge from `min_feasible_ramp_minutes` — see the sensitivity notebook's findings. If the self-consistent fastest ramp came out above 480 minutes, the 8-hour commitment would be physically unreachable regardless of coil size, pad or no pad.

In [3]:
fastest = {
    summer: float(physics.fastest_ramp_minutes(chamber_base.replace(boundary_temp_summer=summer)))
    for summer in (32.0, 45.0)
}
film_ok = {
    summer: (
        bool(physics.compute(chamber_base.replace(boundary_temp_summer=summer, equipment_w_per_m2=0.0))["film_ok"]),
        bool(physics.compute(chamber_base.replace(boundary_temp_summer=summer, equipment_w_per_m2=chamber_base.equipment_w_per_m2))["film_ok"]),
    )
    for summer in (32.0, 45.0)
}
for summer in (32.0, 45.0):
    print(f"summer={summer:.0f}C  fastest self-consistent ramp = {fastest[summer]:.1f} min "
          f"(vs the 480 min committed)   film_ok at 480 min: heating={film_ok[summer][0]}, cooling={film_ok[summer][1]}")

summer=32C  fastest self-consistent ramp = 70.3 min (vs the 480 min committed)   film_ok at 480 min: heating=True, cooling=True
summer=45C  fastest self-consistent ramp = 70.3 min (vs the 480 min committed)   film_ok at 480 min: heating=True, cooling=True


Comfortably inside the film limit: the pad asks for a self-consistent ramp of about 70 minutes, well short of the 8 hours on offer. The 480-minute commitment has plenty of slack against the film constraint, so there is room to add more exposed mass than 12 m² before the pad becomes the binding constraint, if that turns out to matter for other reasons.

**Which mode sets the chamber's size (2026-09-23).** Heating is set by the ramp: with the Sun off in both modes, the ramp adds the mass charge on top of the same hold. Cooling is now set by the *operating* mode, narrowly: holding at 0 °C with the Sun on (≈ 35.9 kW at 45 °C) beats cooling down with the Sun off (≈ 35.4 kW). Under the old hold-plus-ramp sum, cooling was 47.4 kW; it is now 35.9 kW, about 11.5 kW less, because the Sun's 10.5 kW and the mass charge are never asked for at the same time. Heating did not move (35.4 kW), because it was already a Sun-off case.

In [4]:
COMPONENT_COLOR = {
    "Envelope": EARTH[0],
    "Ventilation": EARTH[3],
    "Mass ramp": EARTH[2],
    "Solar": EARTH[4],
    "Internal gain": EARTH[1],
}

def stack(ax, x, components, labelled):
    bottom = 0.0
    for name, value in components.items():
        show = name not in labelled
        ax.bar(x, value, bottom=bottom, color=COMPONENT_COLOR[name], width=0.36,
               label=name if show else None)
        labelled.add(name)
        bottom += value
    return bottom

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)

for ax, summer, title in zip(axes, (45.0, 32.0), ("Summer 45°C (design condition)", "Summer 32°C (superseded reference)")):
    heating = physics.compute(chamber_base.replace(boundary_temp_summer=summer, equipment_w_per_m2=0.0))
    cooling = physics.compute(chamber_base.replace(boundary_temp_summer=summer, equipment_w_per_m2=chamber_base.equipment_w_per_m2))
    labelled = set() if ax is axes[0] else set(COMPONENT_COLOR)

    # Operating: the steady hold with the experiment's own gains.
    # Ramp: the hold at the far setpoint with the Sun off and nobody inside,
    # plus the mass charge. Before margin; the marker is the design value
    # (the larger bar, with margin).
    heat_op = {
        "Envelope": float(kw(heating["envelope_heat"])),
        "Ventilation": float(kw(heating["vent_sensible_heat"])),
    }
    heat_ramp = dict(heat_op, **{"Mass ramp": float(kw(heating["power_mass"]))})
    cool_op = {
        "Envelope": float(kw(cooling["envelope_cool"] - cooling["solar_gain"])),
        "Solar": float(kw(cooling["solar_gain"])),
        "Ventilation": float(kw(cooling["vent_sensible_cool"])),
        "Internal gain": float(kw(cooling["internal_sensible"])),
    }
    cool_ramp = {
        "Envelope": cool_op["Envelope"],
        "Solar": cool_op["Solar"],
        "Ventilation": cool_op["Ventilation"],
        "Internal gain": float(kw(cooling["ramp_internal_sensible"])),
        "Mass ramp": float(kw(cooling["power_mass"])),
    }

    positions = {"heat_op": -0.2, "heat_ramp": 0.2, "cool_op": 0.8, "cool_ramp": 1.2}
    stack(ax, positions["heat_op"], heat_op, labelled)
    stack(ax, positions["heat_ramp"], heat_ramp, labelled)
    stack(ax, positions["cool_op"], cool_op, labelled)
    stack(ax, positions["cool_ramp"], cool_ramp, labelled)

    for duty, r, (op_x, ramp_x) in (("heating", heating, (-0.2, 0.2)), ("cooling", cooling, (0.8, 1.2))):
        x = ramp_x if bool(r[f"{duty}_set_by_ramp"]) else op_x
        design = float(kw(r[f"{duty}_design"]))
        ax.plot([x - 0.2, x + 0.2], [design, design], color=INK, linewidth=1.2, linestyle=":")
        ax.annotate(f"design {design:.1f} kW", (x, design), ha="center", va="bottom", fontsize=7)

    ax.set_xticks(list(positions.values()))
    ax.set_xticklabels(["operating", "ramp", "operating", "ramp"])
    ax.text(0.0, -0.16, "Heating (Sun off)", transform=ax.get_xaxis_transform(), ha="center", fontsize=8)
    ax.text(1.0, -0.16, "Cooling (Sun on while operating)", transform=ax.get_xaxis_transform(), ha="center", fontsize=8)
    ax.set_title(title)
    ax.set_ylabel("kW" if ax is axes[0] else "")

axes[0].legend(loc="upper left", bbox_to_anchor=(0, -0.2), ncol=5, fontsize=7.5)
fig.suptitle("Climate chamber worst case — each mode by component (dotted = design value incl. margin, the larger mode)")
save(fig, 1, "chamber_worst_case_components")
plt.show()

wrote notebooks/figures/fig_1_chamber_worst_case_components.pdf


## Module room: capacity vs ramp time, worst case over the facade

The facade is genuinely still a scenario, so unlike the chamber, this isn't a single number — it's a curve, and the curve has to be the worst case *across every facade the room could end up with*, not just one arbitrarily chosen construction.

### Why the worst-case facade is a corner, not a search

Every facade term enters `heating_design` and `cooling_design` monotonically: more conductance (`south_u_opaque`, `south_u_glazing`, `south_wwr`, `roof_u_opaque`) always increases the design load in both directions, and heating ignores solar entirely (see `physics.py`), so `south_shgc` and `south_irradiance` only ever push `cooling_design` up. That means the worst case is exactly the corner of the box the study config already declares — maximum U-values, maximum WWR, maximum SHGC, maximum irradiance — with no sampling or optimisation needed. A 20 000-point random check below confirms the corner dominates everything sampled inside the box.

In [5]:
FACADE_BOUNDS = {
    "south_u_opaque": (0.05, 2.50),
    "south_u_glazing": (0.50, 3.00),
    "south_wwr": (0.0, 95.0),
    "south_shgc": (0.0, 1.00),
    "south_irradiance": (0.0, 1500.0),
    "roof_u_opaque": (0.05, 2.50),
}
WORST_FACADE = {k: hi for k, (lo, hi) in FACADE_BOUNDS.items()}

module_base = rooms.module_room().replace(
    setpoint_min=18.0, setpoint_max=35.0,
    added_mass_coverage=0.0,
    vent_supply_temp=18.0,
)
module_worst = module_base.replace(**WORST_FACADE)

rng = np.random.default_rng(20260918)
n = 20000
sample = {k: rng.uniform(lo, hi, n) for k, (lo, hi) in FACADE_BOUNDS.items()}
sample_params = module_base.replace(**sample)

for ramp_check in (30.0, 480.0):
    corner = physics.compute(module_worst, ramp_minutes=ramp_check)
    sampled = physics.compute(sample_params, ramp_minutes=ramp_check)
    print(f"ramp={ramp_check:.0f} min   corner heating={float(kw(corner['heating_design'])):.2f} kW "
          f"vs sampled max={float(kw(sampled['heating_design']).max()):.2f} kW   |   "
          f"corner cooling={float(kw(corner['cooling_design'])):.2f} kW "
          f"vs sampled max={float(kw(sampled['cooling_design']).max()):.2f} kW")

ramp=30 min   corner heating=16.50 kW vs sampled max=16.33 kW   |   corner cooling=45.80 kW vs sampled max=43.27 kW
ramp=480 min   corner heating=6.90 kW vs sampled max=6.73 kW   |   corner cooling=36.21 kW vs sampled max=33.67 kW


In [6]:
ramp_minutes = np.geomspace(5.0, 480.0, 60)
worst_case = physics.compute(module_worst, ramp_minutes=ramp_minutes)

fastest_module = float(physics.fastest_ramp_minutes(module_worst))
print(f"Fastest self-consistent ramp at zero added mass: {fastest_module:.1f} min "
      "(facade-independent -- no added mass means nothing for the facade to change)")

crossover = {duty: physics.mode_crossover_minutes(module_worst, duty) for duty in ("heating", "cooling")}
for duty, t in crossover.items():
    print(f"{duty}: operating mode takes over the size from a "
          + (f"{t:.0f} min ramp" if t is not None else "ramp longer than 480 min (never, in range)"))

ramp_curve = pd.DataFrame({
    "ramp_minutes": ramp_minutes,
    "heating_operating_kw": np.broadcast_to(kw(worst_case["heating_operating"]), ramp_minutes.shape),
    "heating_ramp_kw": kw(worst_case["heating_ramp"]),
    "heating_design_kw": kw(worst_case["heating_design"]),
    "cooling_operating_kw": np.broadcast_to(kw(worst_case["cooling_operating"]), ramp_minutes.shape),
    "cooling_ramp_kw": kw(worst_case["cooling_ramp"]),
    "cooling_design_kw": kw(worst_case["cooling_design"]),
    "film_ok": worst_case["film_ok"],
})

Fastest self-consistent ramp at zero added mass: 18.6 min (facade-independent -- no added mass means nothing for the facade to change)
heating: operating mode takes over the size from a ramp longer than 480 min (never, in range)
cooling: operating mode takes over the size from a ramp longer than 480 min (never, in range)


In [7]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(ramp_curve["ramp_minutes"], ramp_curve["heating_design_kw"], color=EARTH[0], linewidth=5, alpha=0.25)
ax.plot(ramp_curve["ramp_minutes"], ramp_curve["cooling_design_kw"], color=EARTH[3], linewidth=5, alpha=0.25)
ax.plot(ramp_curve["ramp_minutes"], ramp_curve["heating_ramp_kw"], color=EARTH[0], marker="o", markersize=3, label="Heating, ramp")
ax.plot(ramp_curve["ramp_minutes"], ramp_curve["heating_operating_kw"], color=EARTH[0], linestyle="--", label="Heating, operating")
ax.plot(ramp_curve["ramp_minutes"], ramp_curve["cooling_ramp_kw"], color=EARTH[3], marker="o", markersize=3, label="Cooling, ramp")
ax.plot(ramp_curve["ramp_minutes"], ramp_curve["cooling_operating_kw"], color=EARTH[3], linestyle="--", label="Cooling, operating")

ax.axvspan(ramp_curve["ramp_minutes"].min(), fastest_module, color=MUTED, alpha=0.25)
ax.text(fastest_module * 1.05, ax.get_ylim()[1] * 0.95, "film-limited\n(unreachable)", va="top", ha="left", fontsize=7.5, color=INK)
ax.axvline(480.0, color=INK, linewidth=1, linestyle=":")
ax.text(480.0, ax.get_ylim()[1] * 0.5, " chamber's 8 h", rotation=90, va="center", fontsize=7.5, color=INK)

ax.set_xscale("log")
ax.set_xlabel("Ramp time (minutes, log scale)")
ax.set_ylabel("Capacity incl. margin (kW)")
ax.set_title("Module room, worst-case facade — operating vs ramp (pale band = design, the larger)")
ax.legend(ncols=2)
save(fig, 2, "module_room_capacity_vs_ramp")
plt.show()

wrote notebooks/figures/fig_2_module_room_capacity_vs_ramp.pdf


### Reading the curve

The ramp requirement falls fast at first and then flattens onto the operating requirement: the mass-charging term (`power_mass = C × ΔT / ramp_seconds`) scales as 1/ramp while the hold underneath it doesn't move. For the module room the two never cross inside 8 hours. The ramp hold differs from the operating hold only by the two occupants, who are not there during a ramp, so as long as the mass charge is larger than their ~0.15 kW the ramp stays the larger mode. In practice, past about 4–5 hours the design value sits within about a kilowatt of the operating hold, so a longer ramp buys almost nothing. Going faster than roughly 20–30 minutes gets expensive quickly, and below the fastest self-consistent ramp printed above it is not reachable at any capacity.

This room's numbers barely moved with the 2026-09-23 change (≈ 0.2 kW, the occupants): it has no Sun to switch off, and the ramp was already setting its size.

In [8]:
representative = [15, 20, 30, 45, 60, 90, 120, 180, 240, 300, 360, 420, 480]
at_points = physics.compute(module_worst, ramp_minutes=np.array(representative, dtype=float))
summary = pd.DataFrame({
    "ramp_minutes": representative,
    "heating_ramp_kw": kw(at_points["heating_ramp"]),
    "heating_design_kw": kw(at_points["heating_design"]),
    "heating_set_by": np.where(at_points["heating_set_by_ramp"], "ramp", "operating"),
    "cooling_ramp_kw": kw(at_points["cooling_ramp"]),
    "cooling_design_kw": kw(at_points["cooling_design"]),
    "cooling_set_by": np.where(at_points["cooling_set_by_ramp"], "ramp", "operating"),
    "film_ok": at_points["film_ok"],
}).set_index("ramp_minutes")
print(f"operating: heating {float(kw(at_points['heating_operating'])):.2f} kW, "
      f"cooling {float(kw(at_points['cooling_operating'])):.2f} kW at every ramp time")
summary.round(2)

operating: heating 6.09 kW, cooling 35.74 kW at every ramp time


,heating_ramp_kw,heating_design_kw,heating_set_by,cooling_ramp_kw,cooling_design_kw,cooling_set_by,film_ok
ramp_minutes,,,,,,,
15,26.74,26.74,ramp,56.04,56.04,ramp,False
20,21.62,21.62,ramp,50.92,50.92,ramp,True
30,16.50,16.50,ramp,45.80,45.80,ramp,True
45,13.09,13.09,ramp,42.39,42.39,ramp,True
60,11.38,11.38,ramp,40.69,40.69,ramp,True
90,9.67,9.67,ramp,38.98,38.98,ramp,True
120,8.82,8.82,ramp,38.13,38.13,ramp,True
180,7.97,7.97,ramp,37.27,37.27,ramp,True
240,7.54,7.54,ramp,36.85,36.85,ramp,True


## Scenario catalog: every combination, ranked by required heat pump capacity

A different question from the two sections above: not "what is the worst case" but "here is the whole space of configurations either room could plausibly end up in -- rank all of them, so any one can be pulled back up later." For the module room that space is the facade construction (still genuinely undecided) crossed with ramp time (also undecided); for the chamber it's the three knobs this session treated as *this session's* worst-case assumptions rather than hard facts -- ACH, pad size, and how cold the pre-conditioned ventilation air is.

**On the file-explosion problem.** The module room's catalog below is 126,720 combinations. Nobody wants 126,720 JSON files, and nobody needs them: a scenario's entire parameter set is reproducible from two things that are already tiny -- the ordered list of (parameter, discrete levels) axes defined below, and one integer. Decoding an id back into levels is exactly the arithmetic numpy already uses to flatten and reshape an array (`np.unravel_index` against the axis shape, i.e. mixed-radix / odometer indexing: the id's low digits cycle through the fastest-moving axis, carrying into the next axis exactly the way seconds carry into minutes). **The id *is* the compressed representation** -- nothing about a scenario is ever written to disk until you specifically ask to export one.

One consequence worth being explicit about: the id-to-parameters mapping depends on the axis definitions below staying fixed. If someone later changes `study/rooms.yaml`'s grid for, say, `south_wwr`, scenario id 84213 will decode to a *different* combination than it does today. That's expected, not a bug -- the id is a key into *this notebook's* current catalog definition, not a portable hash of the physics. Re-run this notebook after any such config change before trusting an old id.

In [9]:
def axis_shape(axes):
    return tuple(len(levels) for _, levels in axes)

def axis_count(axes):
    return int(np.prod(axis_shape(axes)))

def decode_scenario(axes, scenario_id):
    """scenario id -> {parameter: value}, via mixed-radix (odometer) indexing.

    The inverse of generate_catalog's flattening: np.unravel_index is the same
    operation numpy uses internally whenever it reshapes or flattens an array,
    applied here to a virtual grid that is never actually built in memory.
    """
    shape = axis_shape(axes)
    total = int(np.prod(shape))
    if not (0 <= scenario_id < total):
        raise ValueError(f"scenario_id {scenario_id} is out of range for this catalog (0..{total - 1})")
    idx = np.unravel_index(scenario_id, shape)
    return {key: float(np.asarray(levels, dtype=float)[i]) for (key, levels), i in zip(axes, idx)}

def generate_catalog(base_params, axes):
    """Every scenario at once, fully vectorised -- one physics.compute() call

    regardless of how large the catalog is, following the same pattern
    sensitivity.py already uses for its Sobol samples: build a plain dict from
    the baseline and overwrite the varying keys with arrays.
    """
    shape = axis_shape(axes)
    n = int(np.prod(shape))
    ids = np.arange(n)
    idx = np.unravel_index(ids, shape)
    p = dict(base_params.to_dict())
    for (key, levels), ind in zip(axes, idx):
        p[key] = np.asarray(levels, dtype=float)[ind]
    return ids, p

def rank_catalog(ids, results_dict):
    table = pd.DataFrame({
        "scenario_id": ids,
        "heating_kw": kw(results_dict["heating_kw"]),
        "cooling_kw": kw(results_dict["cooling_kw"]),
    })
    table["max_kw"] = table[["heating_kw", "cooling_kw"]].max(axis=1)
    table["binding"] = np.where(table["heating_kw"] >= table["cooling_kw"], "heating", "cooling")
    # Which mode sets the binding duty: the ramp, or holding an experiment.
    if "heating_by_ramp" in results_dict:
        by_ramp = np.where(table["binding"] == "heating",
                           np.broadcast_to(results_dict["heating_by_ramp"], table.shape[:1]),
                           np.broadcast_to(results_dict["cooling_by_ramp"], table.shape[:1]))
        table["set_by"] = np.where(by_ramp, "ramp", "operating")
    return table.sort_values("max_kw", ascending=False).reset_index(drop=True)

def ranked_bar_chart(table, *, title, fig_index, fig_slug, top_n=None):
    shown = table.head(top_n) if top_n else table
    shown = shown.iloc[::-1]  # so #1 plots at the top
    fig, ax = plt.subplots(figsize=(7.5, max(3.5, 0.28 * len(shown))))
    colors = [EARTH[0] if b == "heating" else EARTH[3] for b in shown["binding"]]
    ax.barh(shown["scenario_id"].astype(str), shown["max_kw"], color=colors)
    ax.set_xlabel("Required heat pump capacity, max(heating, cooling) design kW")
    ax.set_ylabel("Scenario id")
    ax.set_title(title)
    from matplotlib.patches import Patch
    ax.legend(
        handles=[Patch(color=EARTH[0], label="heating-bound"), Patch(color=EARTH[3], label="cooling-bound")],
        loc="lower right",
    )
    save(fig, fig_index, fig_slug)
    plt.show()

### Module room catalog: facade × ramp time

Axes come straight from `study/rooms.yaml`'s own grid definitions, already vetted rather than arbitrary -- `south_u_opaque`, `south_u_glazing`, `south_wwr`, `south_shgc` and `roof_u_opaque` are real facade design choices; `ramp_minutes` is the yaml's existing 11-point list. `south_irradiance` is deliberately held fixed at the room's measured south-facing peak (1500 W/m²) -- it's a site condition, not something a facade design chooses -- and ACH/occupants/equipment stay at their rooms.py baseline, since this session didn't ask to explore those.

In [10]:
config = study.load(REPO / "study" / "rooms.yaml")
module_specs = config.rooms["module_room"].specs

MODULE_AXES = [
    ("south_u_opaque", module_specs["south_u_opaque"].grid_points),
    ("south_u_glazing", module_specs["south_u_glazing"].grid_points),
    ("south_wwr", module_specs["south_wwr"].grid_points),
    ("south_shgc", module_specs["south_shgc"].grid_points),
    ("roof_u_opaque", module_specs["roof_u_opaque"].grid_points),
    ("ramp_minutes", module_specs["ramp_minutes"].grid_points),
]
print(f"Module room catalog: {axis_count(MODULE_AXES):,} scenarios "
      f"({' x '.join(f'{k}={len(v)}' for k, v in MODULE_AXES)})")

module_ids, module_p = generate_catalog(module_base, MODULE_AXES)
module_r = physics.compute(module_p)
module_catalog = rank_catalog(module_ids, {
    "heating_kw": module_r["heating_design"], "cooling_kw": module_r["cooling_design"],
    "heating_by_ramp": module_r["heating_set_by_ramp"], "cooling_by_ramp": module_r["cooling_set_by_ramp"],
})
module_catalog.head(10)

Module room catalog: 126,720 scenarios (south_u_opaque=8 x south_u_glazing=6 x south_wwr=6 x south_shgc=5 x roof_u_opaque=8 x ramp_minutes=11)


,scenario_id,heating_kw,cooling_kw,max_kw,binding,set_by
0,126709,36.976322,66.281882,66.281882,cooling,ramp
1,110869,36.959448,66.271286,66.271286,cooling,ramp
2,95029,36.942573,66.260690,66.260690,cooling,ramp
3,79189,36.925698,66.250094,66.250094,cooling,ramp
4,63349,36.908823,66.239498,66.239498,cooling,ramp
5,47509,36.891948,66.228902,66.228902,cooling,ramp
6,31669,36.875073,66.218307,66.218307,cooling,ramp
7,15829,36.858199,66.207711,66.207711,cooling,ramp
8,126698,36.684258,66.098492,66.098492,cooling,ramp
9,110858,36.667383,66.087897,66.087897,cooling,ramp


In [11]:
ranked_bar_chart(
    module_catalog, top_n=25, fig_index=3, fig_slug="module_room_scenario_ranking",
    title=f"Module room -- top 25 of {axis_count(MODULE_AXES):,} scenarios by required capacity",
)
print("Which mode sets the size, across the whole catalog:")
print(module_catalog["set_by"].value_counts().to_string())
print("Full-catalog distribution of max(heating, cooling) kW:")
module_catalog["max_kw"].describe(percentiles=[0.05, 0.5, 0.95]).round(2)

wrote notebooks/figures/fig_3_module_room_scenario_ranking.pdf
Which mode sets the size, across the whole catalog:
set_by
ramp    126720
Full-catalog distribution of max(heating, cooling) kW:


count    126720.00
mean         19.58
std          12.38
min           2.04
5%            4.86
50%          17.10
95%          42.32
max          66.28
Name: max_kw, dtype: float64

The top of the list is exactly the facade-corner result from the section above, at the shortest available ramp (10 min), consistent with both being the same monotonic argument. Every one of the 126,720 scenarios is ramp-set: the module room never reaches the operating crossover within the declared ramp range. What the catalog adds is the shape of everything *below* the worst case: the median scenario needs about a quarter of the top one's capacity (≈ 17 vs 66 kW), so a heat pump sized for the 95th percentile (≈ 42 kW) rather than the absolute corner is a real, visible trade-off here, not a rounding difference.

### Climate chamber catalog: this session's assumptions, varied

The chamber's envelope, setpoints and ramp are settled -- nothing left to vary there. What this session treated as *its own* worst-case assumptions, rather than confirmed facts, are the three knobs below: ACH (0 up to a generous double-refresh), the concrete pad's area (none up to double the 12 m² used above), and how cold the pre-conditioned ventilation supply runs. These don't come from `study/rooms.yaml` -- its chamber block doesn't sweep any of them at levels relevant to this decision -- so they're defined directly here.

In [12]:
chamber_specs = config.rooms["climate_chamber"].specs

CHAMBER_AXES = [
    ("ach", chamber_specs["ach"].grid_points),
    ("added_mass_coverage", chamber_specs["added_mass_coverage"].grid_points),
    ("vent_supply_temp", chamber_specs["vent_supply_temp"].grid_points),
]
print(f"Climate chamber catalog: {axis_count(CHAMBER_AXES):,} scenarios "
      f"({' x '.join(f'{k}={len(v)}' for k, v in CHAMBER_AXES)})")

# The chamber's own worst-case base, but WITHOUT fixing the three knobs above --
# those become the scenario axes instead of a single settled value.
chamber_scenario_base = rooms.climate_chamber().replace(
    setpoint_min=0.0, setpoint_max=45.0,
    ramp_minutes=480.0,
    occupants=0.0,
    added_mass_thickness=0.10,
    added_mass_rho_c=concrete["rho_c"],
    added_mass_k=concrete["k"],
)

# Heating (Sun off) and cooling (Sun on) are still two different scenarios for
# this room, same as everywhere else in this notebook -- generate both, same
# ids both times since the axes and their order are identical.
chamber_ids, chamber_p_heat = generate_catalog(chamber_scenario_base.replace(equipment_w_per_m2=0.0), CHAMBER_AXES)
_, chamber_p_cool = generate_catalog(chamber_scenario_base.replace(equipment_w_per_m2=chamber_scenario_base.equipment_w_per_m2), CHAMBER_AXES)
chamber_r_heat = physics.compute(chamber_p_heat)
chamber_r_cool = physics.compute(chamber_p_cool)
chamber_catalog = rank_catalog(chamber_ids, {
    "heating_kw": chamber_r_heat["heating_design"], "cooling_kw": chamber_r_cool["cooling_design"],
    "heating_by_ramp": chamber_r_heat["heating_set_by_ramp"], "cooling_by_ramp": chamber_r_cool["cooling_set_by_ramp"],
})
chamber_catalog

Climate chamber catalog: 36 scenarios (ach=1 x added_mass_coverage=6 x vent_supply_temp=6)


,scenario_id,heating_kw,cooling_kw,max_kw,binding,set_by
0,30,49.180753,46.253078,49.180753,heating,ramp
1,31,48.695095,46.738735,48.695095,heating,ramp
2,35,46.752464,48.681366,48.681366,cooling,ramp
3,32,48.209437,47.224393,48.209437,heating,ramp
4,34,47.238122,48.195709,48.195709,cooling,ramp
5,33,47.723779,47.710051,47.723779,heating,ramp
6,24,45.802318,42.874644,45.802318,heating,ramp
7,25,45.316661,43.360301,45.316661,heating,ramp
8,29,43.374030,45.302932,45.302932,cooling,ramp
9,26,44.831003,43.845959,44.831003,heating,ramp


In [13]:
ranked_bar_chart(
    chamber_catalog, fig_index=4, fig_slug="chamber_scenario_ranking",
    title=f"Climate chamber -- all {axis_count(CHAMBER_AXES)} scenarios by required capacity",
)

wrote notebooks/figures/fig_4_chamber_scenario_ranking.pdf


As of 2026-09-21 `study/rooms.yaml` holds the chamber's ACH at 1.0 and narrows added-mass coverage to 0–10 %, so this catalog is now 36 scenarios (1 × 6 × 6), not 396; the earlier 41–281 kW range and the heating-bound high-ACH corner no longer exist in it. Across what remains the range is about 34–49 kW. Coverage is the main lever: every extra 2 % of lined surface adds roughly 3 kW to the heating ramp. The ventilation supply temperature moves it by a kilowatt or two either way.

The split by mode is the useful part. With no lining, cooling while *operating* with the Sun on always sets the size; at 2 % coverage it still does in most cases. From about 4 % coverage on, the mass charge is large enough that the *ramp* takes over, and heating becomes the binding duty.

### Pulling any scenario back up by its id

This is the other half of the id-as-key idea: given just a room and an id, reconstruct the full 71-parameter set, recompute its sizing to confirm it matches the catalog row, and print JSON in exactly the shape the browser tool's own Load button reads (`web/app.js`: `payload.params || payload` -- it accepts a `{"params": {...}}` wrapper or a flat dict, either way pulling out one field per matching key). Save the output to a `.json` file and it drops straight into the app with no conversion step.

In [14]:
CATALOGS = {
    "module_room": {"axes": MODULE_AXES, "base": module_base},
    "climate_chamber": {"axes": CHAMBER_AXES, "base": chamber_scenario_base},
}

def describe_scenario(room_key, scenario_id):
    """The whole point: reconstruct a scenario from nothing but its id.

    No file, no database row -- just the axis definitions above (already in
    memory) plus arithmetic. Recomputes both design capacities directly, so
    this never has to trust that a catalog DataFrame is still around or still
    correct.
    """
    spec = CATALOGS[room_key]
    overrides = decode_scenario(spec["axes"], scenario_id)
    params = spec["base"].replace(**overrides)
    if room_key == "climate_chamber":
        heating = physics.compute(params.replace(equipment_w_per_m2=0.0))
        cooling = physics.compute(params.replace(equipment_w_per_m2=spec["base"].equipment_w_per_m2))
    else:
        heating = cooling = physics.compute(params)
    return {
        "room": room_key,
        "scenario_id": scenario_id,
        "overrides": overrides,
        "params": params.to_dict(),
        "heating_design_kw": float(kw(heating["heating_design"])),
        "cooling_design_kw": float(kw(cooling["cooling_design"])),
    }

def scenario_to_json(room_key, scenario_id):
    info = describe_scenario(room_key, scenario_id)
    return json.dumps({"params": info["params"]}, indent=2) + "\n"

best_module_id = int(module_catalog.iloc[0]["scenario_id"])
best_chamber_id = int(chamber_catalog.iloc[0]["scenario_id"])

EXPORT_DIR = REPO / "notebooks" / "scenario_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for room_key, sid in [("module_room", best_module_id), ("climate_chamber", best_chamber_id)]:
    info = describe_scenario(room_key, sid)
    print(f"{room_key} scenario #{sid}: {info['overrides']}")
    catalog = module_catalog if room_key == "module_room" else chamber_catalog
    row = catalog.loc[catalog["scenario_id"] == sid].iloc[0]
    agrees = abs(info["heating_design_kw"] - row["heating_kw"]) < 0.01 and abs(info["cooling_design_kw"] - row["cooling_kw"]) < 0.01
    print(f"  heating {info['heating_design_kw']:.2f} kW   cooling {info['cooling_design_kw']:.2f} kW   (catalog agrees: {agrees})")
    path = EXPORT_DIR / f"{room_key}_scenario_{sid}.json"
    path.write_text(scenario_to_json(room_key, sid))
    print("  wrote", path.relative_to(REPO))

module_room scenario #126709: {'south_u_opaque': 2.4999999999999996, 'south_u_glazing': 3.0, 'south_wwr': 95.0, 'south_shgc': 1.0, 'roof_u_opaque': 2.4999999999999996, 'ramp_minutes': 10.0}
  heating 36.98 kW   cooling 66.28 kW   (catalog agrees: True)
  wrote notebooks/scenario_exports/module_room_scenario_126709.json
climate_chamber scenario #30: {'ach': 1.0, 'added_mass_coverage': 10.0, 'vent_supply_temp': 12.0}
  heating 49.18 kW   cooling 46.25 kW   (catalog agrees: True)
  wrote notebooks/scenario_exports/climate_chamber_scenario_30.json


Exactly two files on disk, whatever the catalog size -- the top-ranked scenario for each room, ready to load into the browser tool (`python -m zcbsl_resize.server`, then the Load button). Any other id from either bar chart above reconstructs the same way: `scenario_to_json("module_room", 84213)` and save the result, no need to have exported it in advance.

## Open items this notebook surfaces but doesn't settle

- **`boundary_temp_summer` is now resolved at 45°C** -- confirmed as deliberate climate-change future-proofing, and `rooms.py` carries it directly as of this session. The 32°C column in the chart above is kept only as a superseded historical reference, not a live option.
- **Concrete pad thickness (100 mm)** was this session's answer, not a measured value — if the real pad ends up thicker, rerun the chamber cell with `added_mass_thickness` changed; the pad is nowhere near the film limit at 8 h so there's real headroom to increase it.
- **Resolved 2026-09-18: added mass is a percentage of interior area, not an absolute m², everywhere.** `added_mass_area` no longer exists as a settable input at all -- `added_mass_coverage` (percent of interior surface) is the only way to set it, and `physics.compute()` converts it to an area internally, using each room's own interior surface. Both rooms' `study/rooms.yaml` grids now share the identical 0-100% range, which is what makes the chamber catalog below pull its axes straight from the config instead of hardcoding them.
- **Module room ramp time is still a decision, not a finding.** This notebook gives you the trade curve, not the answer — 8 h matches the chamber's commitment but buys little beyond 2-4 h; something shorter is where the curve actually costs you.
- **`vent_supply_temp=18°C` sits below `surrounding_temp=21°C`** for both rooms — pre-conditioned supply air colder than the lab it sits in. That may be deliberate (conditioning margin) but is worth confirming rather than assuming; if it should track the lab instead, it changes the ventilation term in both rooms' hold loads.
- **Decided 2026-09-23: design capacity is the larger of operating and ramp, never the sum; ramps run with the Sun off and nobody inside.** This removed about 11.5 kW from the chamber's worst-case cooling and left heating and the module room essentially unchanged. The browser tool shows both modes side by side, and where the operating mode takes over.
- **Latent load is still outside the cooling capacity.** `latent_design` (≈ 3.6 kW for the chamber here) is reported separately; a coil that also dehumidifies has to carry it on top.